In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

remove_attention_failers = 0 #Remove participants who failed attention checks (deprecated for first study as they are all replaced)

In [2]:

def extract_basic_info(csv_path):
    df = pd.read_csv(csv_path)


    # Food reported across training trials
    train_responses = df['slider_train.response'].dropna().tolist()

    #Actual food needed (food needed on each training trial)
    fertility_score = df['fertility_score'].dropna().tolist() [:-1]
    print(fertility_score)
    
    # Feedback (free text response about experiment)
    col = 'feedback_text.text'

    if col not in df.columns:
        warnings.warn(
            f"Missing column '{col}' in {os.path.basename(csv_path)}"
        )
        feedback = []
    else:
        feedback = df[col].dropna().tolist()
    
    # Trial stop time (time it took to finish the training loop)
    isi_values = df['ISI.stopped'].dropna().tolist()

    #Get the ISI value for the last training trial, store it
    trial_stop_time = isi_values[-1] if isi_values else np.nan

    #First row with a non-empty value in 'images_list', which shows the order of testing images presented
    images_row = df[df['images_list'].notna()].iloc[0] if not df[df['images_list'].notna()].empty else None

    #Turn the images from PNGs to names
    images = [img.split('/')[-1].replace('.png','') for img in images_row['images_list'].split(',')]

    #First row with a non-empty value in 'sliderRatings', which shows the ratings for testing images
    ratings_row = df[df['sliderRatings'].notna()].iloc[0] if not df[df['sliderRatings'].notna()].empty else None

    #Turn them into floats split by commas
    ratings = [float(r) for r in ratings_row['sliderRatings'].strip('[]').split(',')]

    #Across the training trials, add information about what feature was presented
    train_feet = df['feet'].dropna().tolist() [:-1]
    train_stripes = df['stripes'].dropna().tolist() [:-1]

    #Across the training trials, add information about feature relevance
    train_categories =  df['category'].dropna().tolist() [:-1]

    #Extracting the relevant and irrelevant feature dimension info
    dims = {}
    cols = ['relevant_dim', 'irrelevant_dim', 
            'feet_high', 'stripes_low', 
            'stripes_high', 'feet_low']

    for col in cols:
        vals = df[col].dropna().unique()

        if len(vals) == 0:
            dims[col] = np.nan
            print(f"Warning: No values found in {col}")

        elif len(vals) == 1:
            dims[col] = vals[0]

        else:
            warnings.warn(
                f"Multiple values found in {col}: {vals}"
            )
            dims[col] = vals[0] 

    print(dims)
    
    #Categories for the testing images, in the order shown
    test_categories = images_row['testing_categories'].split(',')
    
    #Write condition (this is the unique identifier for a certain order of trials)
    condition = images_row['condition'] if images_row is not None and 'condition' in images_row else np.nan

    #Add in the order of images during training
    training_image_order = [img.split('/')[-1].replace('.png','') 
                        for img in df['image_file'].dropna().tolist()] [:-1]


    #Updated code to get slider responses (subjective reports of feature relevance)
    df['feature_clean'] = (
        df['feature']
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({'stripe': 'stripes'})
    )
    slider_responses = {}
    features = ['feet', 'stripes']

    for feat in features:
        sub = df[df['feature_clean'] == feat]

        for suffix in ['', '_mid']:

            # 1. discrete
            disc_col = f'discrete_slider{suffix}.response'
            disc = sub[disc_col]

            disc_val = (
                disc[disc.notna() & (disc != "")].iloc[0]
                if (disc.notna() & (disc != "")).any()
                else np.nan
            )

            slider_responses[f'{feat}_discrete_slider{suffix}.response'] = disc_val

            # 2. direction
            dir_col = f'direction_response_label{suffix}'

            if dir_col not in sub.columns:
                warnings.warn(
                    f"Column '{dir_col}' not found for feature '{feat}' in {os.path.basename(csv_path)}"
                )
                dirc_val = np.nan
            else:
                dirc = sub[dir_col]
                dirc_val = (
                    dirc[dirc.notna() & (dirc != "")].iloc[0]
                    if (dirc.notna() & (dirc != "")).any() and disc_val != 'No'
                    else np.nan
                )

            slider_responses[f'{feat}_direction_slider{suffix}.response'] = dirc_val

            # 3. continuous
            cont_col = f'continuous_slider{suffix}.response'
            cont = sub[cont_col]

            cont_val = (
                cont[cont.notna() & (cont != "")].iloc[0]
                if (cont.notna() & (cont != "")).any() and disc_val != 'No'
                else np.nan
            )

            slider_responses[f'{feat}_continuous_slider{suffix}.response'] = cont_val

    #Adding attention check result
    att_row = df['answer_3_right.numClicks'].dropna()
    print('attention', att_row)
    if not att_row.empty:
        att_row = int(att_row.iloc[0])
    else:
        warnings.warn("No attention check data found.")

    print('attention', att_row)
    print('type', type(att_row))

    # Convert to pass/fail (1 = correct, 0 = incorrect)
    attention_check = 1 if att_row == 1 else 0

    #Spontaneous explain/predict ratings:
    cogpro_predict = df['cogpro_predict.response'].dropna().iloc[0]
    cogpro_explain = df['cogpro_explain.response'].dropna().iloc[0]
    cogpro_predict = float(pd.Series([cogpro_predict]).astype(str).str.extract(r'(\d+\.?\d*)').iloc[0,0])
    cogpro_explain = float(pd.Series([cogpro_explain]).astype(str).str.extract(r'(\d+\.?\d*)').iloc[0,0])

    result = {
        'participant': os.path.basename(csv_path)[:3], #participant number
        'training_responses': train_responses, #Response during training 
        'fertility_score': fertility_score, #real fertility score displayed that trial
        'error': [abs(pred - actual) for pred, actual in zip(train_responses, fertility_score)],
        'feedback': feedback,
        'trial_stop_time': trial_stop_time,
        'testing_image_order': images,
        'testing_responses': ratings,
        'training_categories': train_categories,
        'training_feet': train_feet,
        'training_stripes': train_stripes,
        'testing_categories': test_categories,
        'conditionOrder': condition, 
        'training_image_order': training_image_order,
        'attention_check': attention_check,
        'cogpro_predict': cogpro_predict,
        'cogpro_explain': cogpro_explain,
        **dims
    }
    result.update(slider_responses)
    return result

topdir = '/Users/sm6511/Desktop/Prediction-Accomodation-Exp'
study = 'study5.0'
dates = [
    '2026-06-04',
    '2026-06-08',
    '2026-06-10'
]
datadir = os.path.join(topdir, f'data/{study}/Predict')
cleaneddir = os.path.join(topdir, f'data/{study}/Cleaned')
all_participants = []

for fname in os.listdir(datadir):
    if fname.endswith('.csv') and fname:
        participant_id = fname[:3]
        if not any(d in fname for d in dates):
            continue
        csv_path = os.path.join(datadir, fname)
        print(csv_path)
        info = extract_basic_info(csv_path)
        all_participants.append(info)


df_all = pd.DataFrame(all_participants)
if remove_attention_failers:
    df_all = df_all[df_all['attention_check'] == 1]
    df_all.to_csv(os.path.join(cleaneddir, f'{study}PredictAttRemoved.csv'), index=False)
else:
    df_all.to_csv(os.path.join(cleaneddir, f'{study}Predict.csv'), index=False)

print(df_all[df_all['attention_check'] == 1])

/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/study5.0/Predict/282_test_2026-06-10_13h26.28.430.csv
[5.0, 9.0, 2.0, 8.0, 7.0, 8.0, 4.0, 2.0, 7.0, 3.0, 8.0, 3.0, 1.0, 8.0, 9.0, 3.0]
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'F', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'C'}
attention 30    1.0
Name: answer_3_right.numClicks, dtype: float64
attention 1
type <class 'int'>
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/study5.0/Predict/074_test_2026-06-04_14h46.12.366.csv
[5.0, 5.0, 8.0, 4.0, 8.0, 7.0, 2.0, 3.0, 4.0, 9.0, 5.0, 3.0, 2.0, 2.0, 8.0, 10.0]
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'F', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'C'}
attention 30    1.0
Name: answer_3_right.numClicks, dtype: float64
attention 1
type <class 'int'>
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/study5.0/Predict/211_test_2026-06-08_10h32.40.960.csv
[10.0, 5.0, 7.0, 6.0, 2.0, 9.0, 4.0, 5.0, 10.0, 1.0, 2

/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label_mid' not found for feature 'feet' in 282_test_2026-06-10_13h26.28.430.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label_mid' not found for feature 'stripes' in 282_test_2026-06-10_13h26.28.430.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label_mid' not found for feature 'feet' in 074_test_2026-06-04_14h46.12.366.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label_mid' not found for feature 'stripes' in 074_test_2026-06-04_14h46.12.366.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direc

[8.0, 2.0, 7.0, 2.0, 10.0, 3.0, 10.0, 3.0, 6.0, 2.0, 6.0, 8.0, 10.0, 2.0, 3.0, 8.0]
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'F', 'stripes_low': 'W', 'stripes_high': 'E', 'feet_low': 'C'}
attention 30    1.0
Name: answer_3_right.numClicks, dtype: float64
attention 1
type <class 'int'>
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/study5.0/Predict/242_test_2026-06-04_13h47.32.940.csv
[10.0, 2.0, 2.0, 8.0, 3.0, 7.0, 1.0, 2.0, 2.0, 6.0, 7.0, 9.0, 5.0, 5.0, 6.0, 5.0]
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'C', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'F'}
attention 30    1.0
Name: answer_3_right.numClicks, dtype: float64
attention 1
type <class 'int'>
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/study5.0/Predict/248_test_2026-06-10_14h29.22.066.csv
[5.0, 4.0, 3.0, 5.0, 3.0, 6.0, 6.0, 1.0, 7.0, 4.0, 8.0, 1.0, 6.0, 8.0, 5.0, 5.0]
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'F', 'stripe

/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 142_test_2026-06-04_13h44.16.617.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label' not found for feature 'stripes' in 142_test_2026-06-04_13h44.16.617.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 189_test_2026-06-04_14h52.08.812.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label' not found for feature 'stripes' in 189_test_2026-06-04_14h52.08.812.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_la

[8.0, 8.0, 2.0, 6.0, 4.0, 6.0, 7.0, 4.0, 6.0, 8.0, 5.0, 10.0, 1.0, 8.0, 7.0, 4.0]
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'F', 'stripes_low': 'W', 'stripes_high': 'E', 'feet_low': 'C'}
attention 30    1.0
Name: answer_3_right.numClicks, dtype: float64
attention 1
type <class 'int'>
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/study5.0/Predict/262_test_2026-06-10_14h25.20.969.csv
[2.0, 1.0, 7.0, 8.0, 10.0, 2.0, 1.0, 7.0, 3.0, 10.0, 7.0, 6.0, 2.0, 7.0, 5.0, 2.0]
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'C', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'F'}
attention 30    1.0
Name: answer_3_right.numClicks, dtype: float64
attention 1
type <class 'int'>
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/study5.0/Predict/023_test_2026-06-04_14h43.16.845.csv
[1.0, 2.0, 10.0, 8.0, 6.0, 8.0, 7.0, 7.0, 8.0, 4.0, 7.0, 1.0, 2.0, 8.0, 7.0, 10.0]
{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'F', 'strip

...

/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 243_test_2026-06-10_14h28.17.066.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label' not found for feature 'stripes' in 243_test_2026-06-10_14h28.17.066.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label_mid' not found for feature 'feet' in 014_test_2026-06-04_14h41.56.165.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label_mid' not found for feature 'stripes' in 014_test_2026-06-04_14h41.56.165.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_res

[9.0, 3.0, 10.0, 7.0, 1.0, 6.0, 2.0, 7.0, 3.0, 10.0, 7.0, 4.0, 6.0, 4.0, 10.0, 6.0]
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'F', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'C'}
attention 30    1.0
Name: answer_3_right.numClicks, dtype: float64
attention 1
type <class 'int'>
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/study5.0/Predict/155_test_2026-06-04_14h44.32.858.csv
[10.0, 3.0, 4.0, 7.0, 9.0, 6.0, 5.0, 10.0, 8.0, 4.0, 8.0, 4.0, 9.0, 5.0, 1.0, 6.0]
{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'F', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'C'}
attention 30    1.0
Name: answer_3_right.numClicks, dtype: float64
attention 1
type <class 'int'>
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/study5.0/Predict/287_test_2026-06-04_13h49.43.235.csv
[4.0, 5.0, 4.0, 7.0, 2.0, 9.0, 9.0, 4.0, 4.0, 1.0, 4.0, 8.0, 10.0, 2.0, 2.0, 5.0]
{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'F', 'stri

/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label_mid' not found for feature 'feet' in 214_test_2026-06-04_14h46.01.157.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label_mid' not found for feature 'stripes' in 214_test_2026-06-04_14h46.01.157.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 009_test_2026-06-04_14h41.19.547.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label_mid' not found for feature 'feet' in 009_test_2026-06-04_14h41.19.547.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_re

[3.0, 6.0, 5.0, 3.0, 3.0, 7.0, 7.0, 3.0, 6.0, 6.0, 10.0, 5.0, 6.0, 7.0, 10.0, 5.0]
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'F', 'stripes_low': 'W', 'stripes_high': 'E', 'feet_low': 'C'}
attention 30    1.0
Name: answer_3_right.numClicks, dtype: float64
attention 1
type <class 'int'>
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/study5.0/Predict/009_test_2026-06-04_14h41.19.547.csv
[1.0, 5.0, 5.0, 3.0, 1.0, 4.0, 10.0, 6.0, 7.0, 9.0, 5.0, 5.0, 5.0, 10.0, 4.0, 4.0]
{'relevant_dim': 'feet', 'irrelevant_dim': 'stripes', 'feet_high': 'C', 'stripes_low': 'E', 'stripes_high': 'W', 'feet_low': 'F'}
attention 30    1.0
Name: answer_3_right.numClicks, dtype: float64
attention 1
type <class 'int'>
/Users/sm6511/Desktop/Prediction-Accomodation-Exp/data/study5.0/Predict/004_test_2026-06-04_14h41.30.218.csv
[4.0, 9.0, 3.0, 3.0, 10.0, 8.0, 6.0, 5.0, 8.0, 6.0, 5.0, 5.0, 3.0, 8.0, 8.0, 4.0]
{'relevant_dim': 'stripes', 'irrelevant_dim': 'feet', 'feet_high': 'F', 'strip

/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label' not found for feature 'feet' in 042_test_2026-06-04_11h42.43.971.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label' not found for feature 'stripes' in 042_test_2026-06-04_11h42.43.971.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label_mid' not found for feature 'feet' in 247_test_2026-06-04_14h47.09.652.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_response_label_mid' not found for feature 'stripes' in 247_test_2026-06-04_14h47.09.652.csv
  warnings.warn(
/var/folders/t5/wz7tq5fx44j9z9q48fv6hc0c0000gq/T/ipykernel_70722/1667434640.py:115: UserWarning: Column 'direction_res

In [3]:
print(np.sort(df_all['conditionOrder']))

[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  77  78  79  80  81  82  83  84  85  86  87  88  89  90  91
  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108 109
 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127
 128 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146
 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161 162 163 164
 165 166 167 168 169 170 171 172 173 174 175 176 178 179 180 181 182 183
 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201
 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219
 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237
 238 239 241 242 243 244 245 246 247 248 249 250 25

In [24]:
print(len(df_all['training_categories'][0]))

24


In [5]:
print(df_all['testing_categories'])

0    [medium, low, medium, medium, high, high, medi...
1    [high, medium, low, medium, medium, high, low,...
2    [medium, medium, high, low, high, medium, low,...
3    [medium, high, high, low, medium, medium, low,...
Name: testing_categories, dtype: object
